In [1]:
import os
import sys
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.rag.document_chunker import HybridDocumentChunker

# 1. تهيئة المسارات وتحميل الكلاس
data_path = "../data_Json/processed/master_scholarships_clean.json"
chunker = HybridDocumentChunker(chunk_size=500, chunk_overlap=50)

# 2. تنفيذ الخطوة الأولى: التحميل والتصفية
print(">>> Loading and Filtering Documents...")
base_docs = chunker.load_and_filter(data_path)
print(f"Loaded {len(base_docs)} base documents successfully.")

# 3. تنفيذ الخطوة الثانية: التقطيع الهجين
print(">>> Applying Hybrid Chunking...")
final_chunks = chunker.process_documents(base_docs)
print(f"Generated {len(final_chunks)} total chunks.")

# 4. التحليل الهندسي المتقدم (Advanced Diagnostics)
print("\n--- Chunking Quality Diagnostics ---")

# حساب أطوال القطع
chunk_lengths = [len(chunk.page_content) for chunk in final_chunks]
df_stats = pd.Series(chunk_lengths)

print("Character Length Distribution:")
print(df_stats.describe())

# فحص القطع المتيتّمة (Chunks missing crucial semantic context)
orphan_chunks = 0
for chunk in final_chunks:
    if 'Document_Title' not in chunk.metadata and 'Section_Title' not in chunk.metadata:
        orphan_chunks += 1

print(f"\nOrphan Chunks (Missing Contextual Headers): {orphan_chunks}")

# فحص تسوية البيانات الوصفية (Metadata Flattening Validation)
nested_metadata = 0
for chunk in final_chunks:
    if any(isinstance(val, dict) for val in chunk.metadata.values()):
        nested_metadata += 1

print(f"Chunks with nested metadata (ChromaDB Crash Risk): {nested_metadata}")

# 5. عرض عينة عشوائية لفحص جودة التقطيع
import random
print("\n>>> Random Chunk Inspection <<<")
sample_chunk = random.choice(final_chunks)
print("METADATA:")
for k, v in sample_chunk.metadata.items():
    print(f"  {k}: {v}")
print("\nCONTENT:")
print("-" * 50)
print(sample_chunk.page_content)
print("-" * 50)

>>> Loading and Filtering Documents...
Loaded 2526 base documents successfully.
>>> Applying Hybrid Chunking...
Generated 20440 total chunks.

--- Chunking Quality Diagnostics ---
Character Length Distribution:
count    20440.000000
mean       357.842319
std        123.387443
min         39.000000
25%        269.000000
50%        399.000000
75%        461.000000
max        500.000000
dtype: float64

Orphan Chunks (Missing Contextual Headers): 0
Chunks with nested metadata (ChromaDB Crash Risk): 0

>>> Random Chunk Inspection <<<
METADATA:
  scholarship_name: In-Region Scholarships for Postgraduate Studies for Yemenis in Jordan (Syria - Doctoral candidates/PhD students)
  host_country: Germany
  eligible_nationality: Syria
  academic_level: Doctoral/PhD
  academic_major: All Disciplines
  funding_category: Fixed Grant
  funding_amount: 600 eur
  standardized_deadline: Not Specified
  scholarship_status: Rolling/Unspecified
  application_link: https://www2.daad.de/deutschland/stipendium/